In [5]:
#creation of acceptance_times.csv

In [6]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, subprocess, shutil

DATA = "/content/drive/MyDrive/S08 NLP/1_2nov/nov2_data"
FILES = "/content/drive/MyDrive/S08 NLP/files csv"
RESULTS = "/content/drive/MyDrive/S08 NLP/results/1_results"
EVENTS = os.path.join(DATA, "event_table.csv")
OUTPUT = os.path.join(FILES, "acceptance_times.csv")

assert os.path.isfile(EVENTS), f"Missing input: {EVENTS}"
os.makedirs(FILES, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print("Input:", EVENTS)
print("Output:", OUTPUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: /content/drive/MyDrive/S08 NLP/1_2nov/nov2_data/event_table.csv
Output: /content/drive/MyDrive/S08 NLP/files csv/acceptance_times.csv


In [7]:
%%writefile fetch_acceptance.py
#we extract ACCEPTANCE-DATETIME directly from each filing header, as the submissions
#JSON is unreliable (201 mismatches out of 300 filings: 128 mislabeled as UTC and
#73 with an incorrect offset). We fetch one header per filing, resume from saved
#results, and save the timestamps in a CSV keyed by accession for build_event_table.py.
#python fetch_acceptance.py --events event_table.csv

from __future__ import annotations

import argparse
import csv
import os
import re
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

USER_AGENT = "MIT 15.S08 project mbonnetderoovere@gmail.com"
RATE = 9.0
WORKERS = 8
HEADER_RE = re.compile(r"<ACCEPTANCE-DATETIME>\s*(\d{14})")


class RateLimiter:
    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)


def fetch_head(url: str, limiter: RateLimiter, retries: int = 3) -> str | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                #the header is the first thing in the submission file.
                return r.read(4096).decode("utf-8", "replace")
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None


def already_done(path: str) -> set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, newline="", encoding="utf-8") as f:
        return {r["accession"] for r in csv.DictReader(f)}


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--events", default="event_table.csv")
    ap.add_argument("--out", default="acceptance_times.csv")
    ap.add_argument("--workers", type=int, default=WORKERS)
    ap.add_argument("--limit", type=int, default=0)
    args = ap.parse_args()

    seen, rows = set(), []
    with open(args.events, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            a = r["accession"]
            if a and a not in seen:
                seen.add(a)
                rows.append((a, r["cik"]))

    done = already_done(args.out)
    fresh = not done
    if done:
        before = len(rows)
        rows = [r for r in rows if r[0] not in done]
        print(f"resuming: {before - len(rows):,} stored, {len(rows):,} to go",
              flush=True)
    if args.limit:
        rows = rows[:args.limit]
    if not rows:
        print("nothing to do")
        return 0

    limiter = RateLimiter(RATE)
    lock = threading.Lock()
    t0 = time.monotonic()
    n = {"done": 0, "missing": 0}

    out = open(args.out, "a", newline="", encoding="utf-8")
    w = csv.writer(out)
    if fresh:
        w.writerow(["accession", "acceptance_header_et"])

    def work(item) -> None:
        acc, cik = item
        url = (f"https://www.sec.gov/Archives/edgar/data/{str(cik).lstrip('0')}/"
               f"{acc.replace('-', '')}/{acc}.txt")
        head = fetch_head(url, limiter)
        m = HEADER_RE.search(head or "")
        stamp = m.group(1) if m else ""
        with lock:
            w.writerow([acc, stamp])
            n["done"] += 1
            if not stamp:
                n["missing"] += 1
            k = n["done"]
            if k % 2000 == 0:
                out.flush()
                el = time.monotonic() - t0
                rate = k / el
                left = (len(rows) - k) / rate / 3600 if rate else 0
                print(f"  {k:,}/{len(rows):,}  {rate:.1f}/s  "
                      f"{left:.1f} h left  {n['missing']:,} without a header",
                      flush=True)

    with ThreadPoolExecutor(max_workers=args.workers) as ex:
        list(ex.map(work, rows))
    out.close()

    print(f"\n{n['done']:,} fetched in {(time.monotonic() - t0) / 3600:.2f} h")
    print(f"{n['missing']:,} with no ACCEPTANCE-DATETIME in the header")
    print(f"-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Overwriting fetch_acceptance.py


In [ ]:
#fetch official EDGAR acceptance times

import subprocess
import sys
import shutil

OUTPUT = "/content/drive/MyDrive/S08 NLP/files csv/acceptance_times.csv"

subprocess.run([
    sys.executable, "fetch_acceptance.py",
    "--events", os.path.join(DATA, "event_table.csv"),
    "--out", OUTPUT
], check=True)

shutil.copy2(
    OUTPUT,
    "/content/drive/MyDrive/S08 NLP/results/1_results/acceptance_times.csv"
)

print("Saved in files csv and 1_results")

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT, dtype={"accession": str})
print("Rows:", len(df))
print("Valid timestamps:", df["acceptance_header_et"].notna().sum())
print("Missing timestamps:", df["acceptance_header_et"].isna().sum())
display(df.head())